# Comprehensive Next-Day AQI Model Evaluation

This notebook compares XGBoost next-day AQI classification under the project’s **6-class, 3-class, and clean-vs-polluted (2-class)** definitions. It reports accuracy, macro F1, the training-majority baseline, and a persistence baseline, plus class-level diagnostics.

The notebook reads the daily `aqi_daily` table from MySQL, uses today’s completed pollutant and weather summaries, and predicts the same station’s next calendar-day AQI class. All three tasks use one shared chronological 80/20 date split. No repository data or database table is modified.

**Interpretation note:** Accuracy can be inflated by common classes, so review macro F1, per-class scores, confusion matrices, and the baselines together. The held-out period is later than the training period; results are experimental, not operational health guidance.


## 1. Imports and project configuration

Load the shared AQI breakpoint functions, modeling libraries, and database settings. Set `DB_URL` in the environment if the local MySQL connection differs from the default.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, brier_score_loss,
    classification_report, confusion_matrix, f1_score,
    log_loss, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, label_binarize
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "air-quality-classification" / "app" / "aqi_labels.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the Data_Science repository or its subfolders.")
APP_DIR = ROOT / "air-quality-classification" / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
from aqi_labels import (
    AQI_CATEGORIES, O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS,
    category_codes, interpolate_iaqi,
)

DB_URL = os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality")
SOURCE_TABLE = "aqi_daily"
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = ["TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
           "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos"]
FEATURES = POLLUTANTS + WEATHER
RANDOM_STATE = 42


## 2. Load the daily database table

Read `aqi_daily` directly from MySQL. Validate required columns and the unique station-day key before continuing. The table is only read.


In [ ]:
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
finally:
    engine.dispose()

required = ["date", "station", "station_id", *POLLUTANTS, "O3_1h_max", *WEATHER]
missing = [column for column in required if column not in source.columns]
if missing:
    raise KeyError(f"air_quality.{SOURCE_TABLE} is missing required columns: {missing}")
if source.duplicated(["station_id", "date"]).any():
    raise ValueError("aqi_daily contains duplicate station-days.")
source["date"] = pd.to_datetime(source["date"], errors="coerce").dt.normalize()
print(f"Loaded {len(source):,} daily station rows")
print(f"Stations: {source.station.nunique()} | Dates: {source.date.min().date()} through {source.date.max().date()}")
display(source.head())


## 3. Rebuild AQI and exact next-day targets

Calculate China AQI sub-indices using the project’s shared breakpoint helper. For ozone above 800 µg/m³, use its daily 1-hour maximum. Join each day’s features to the same station’s following calendar day, so missing dates are not silently treated as consecutive observations.

The class mappings match the existing model notebooks: six official AQI categories; three classes of AQI ≤100, 101–200, and >200; and Clean (AQI ≤100) vs Polluted (AQI >100).


In [ ]:
daily = source.copy()
for column in [*POLLUTANTS, "O3_1h_max", *WEATHER]:
    daily[column] = pd.to_numeric(daily[column], errors="coerce")

iaqi_columns = []
for pollutant in POLLUTANTS:
    values = daily[pollutant]
    iaqi_column = f"{pollutant}_iaqi"
    if pollutant == "O3":
        ozone_8h = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS["O3"])
        ozone_1h = interpolate_iaqi(daily["O3_1h_max"], O3_1H_BREAKPOINTS)
        ozone_8h.loc[values.gt(800)] = ozone_1h.loc[values.gt(800)]
        daily[iaqi_column] = ozone_8h.round()
    else:
        daily[iaqi_column] = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS[pollutant]).round()
    iaqi_columns.append(iaqi_column)

daily["AQI"] = daily[iaqi_columns].max(axis=1, skipna=True)
daily["category_code"] = category_codes(daily["AQI"])
daily["category"] = daily["category_code"].map(AQI_CATEGORIES)

today = daily[["date", "station_id", "station", *FEATURES, "AQI", "category_code"]].copy()
tomorrow = daily[["date", "station_id", "AQI", "category_code"]].copy()
tomorrow["date"] = tomorrow["date"] - pd.Timedelta(days=1)
tomorrow = tomorrow.rename(columns={
    "AQI": "target_AQI", "category_code": "target_category_code"
})
data = today.merge(tomorrow, on=["date", "station_id"], how="inner", validate="one_to_one")
data = data.dropna(subset=["target_AQI", "target_category_code"]).copy()
data["target_category_code"] = data["target_category_code"].astype(int)
data["target_6class"] = data["target_category_code"].map(AQI_CATEGORIES)
data["target_3class"] = np.select(
    [data["target_AQI"] <= 100, data["target_AQI"] <= 200],
    ["Clean / good", "Light-to-moderate polluted"],
    default="Heavily polluted",
)
data["target_2class"] = np.where(data["target_AQI"] <= 100, "Clean", "Polluted")
data["today_6class"] = data["category_code"].map(AQI_CATEGORIES)
data["today_3class"] = np.select(
    [data["AQI"] <= 100, data["AQI"] <= 200],
    ["Clean / good", "Light-to-moderate polluted"],
    default="Heavily polluted",
)
data["today_2class"] = np.where(data["AQI"] <= 100, "Clean", "Polluted")
for today_col in ["today_6class", "today_3class", "today_2class"]:
    data.loc[data["AQI"].isna(), today_col] = np.nan
data = data.sort_values(["date", "station"]).reset_index(drop=True)
print(f"Exact next-calendar-day examples: {len(data):,}")
display(data[["date", "station", "target_AQI", "target_6class", "target_3class", "target_2class"]].head())


## 4. Create one chronological holdout

The same first 80% of unique dates trains every task; the last 20% is held out. Keeping all station rows for a date on one side avoids training on some stations’ readings from a day while testing on other stations’ readings from that same day.


In [ ]:
unique_dates = np.array(sorted(data["date"].dropna().unique()))
cut = int(len(unique_dates) * 0.80)
train_dates, test_dates = unique_dates[:cut], unique_dates[cut:]
train = data[data.date.isin(train_dates)].copy()
test = data[data.date.isin(test_dates)].copy()
print(f"Train: {len(train):,} rows, {train.date.min().date()} through {train.date.max().date()}")
print(f"Test:  {len(test):,} rows, {test.date.min().date()} through {test.date.max().date()}")
print("Features used by all three models:", FEATURES)


## 5. Train and score the 6-, 3-, and 2-class models

Tune XGBoost only within training dates using expanding-window folds and macro F1. The untouched test dates are scored once. For each task, compare the model with (1) a majority baseline that always predicts the most common training class and (2) persistence, which predicts that today’s AQI class will continue tomorrow.

The summary includes accuracy and macro F1 for all three approaches, plus balanced accuracy, macro precision/recall, and probability scores for the trained model. Detailed per-class metrics and confusion matrices follow.


In [ ]:
TASKS = {
    "6-class": ("target_6class", "today_6class",
                [AQI_CATEGORIES[i] for i in range(1, 7)]),
    "3-class": ("target_3class", "today_3class",
                ["Clean / good", "Light-to-moderate polluted", "Heavily polluted"]),
    "2-class": ("target_2class", "today_2class", ["Clean", "Polluted"]),
}

# Date-grouped expanding-window validation folds, built only from training dates.
train_date_positions = pd.factorize(train.date, sort=True)[0]
folds = []
for earlier, later in TimeSeriesSplit(n_splits=4).split(np.unique(train_date_positions)):
    folds.append((np.flatnonzero(np.isin(train_date_positions, earlier)),
                  np.flatnonzero(np.isin(train_date_positions, later))))

PARAM_GRID = {
    "classifier__n_estimators": [120, 240],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.06],
}

models, encoders, predictions, probabilities = {}, {}, {}, {}
summary_rows, reports, confusion_tables = [], {}, {}

for task_name, (target_col, persistence_col, class_names) in TASKS.items():
    task_train = train.dropna(subset=[target_col]).copy()
    task_test = test.dropna(subset=[target_col]).copy()
    encoder = LabelEncoder().fit(task_train[target_col].astype(str))
    y_train = encoder.transform(task_train[target_col].astype(str))
    y_test = encoder.transform(task_test[target_col].astype(str))
    present_classes = np.unique(y_train)
    if len(present_classes) != len(class_names):
        raise ValueError(f"{task_name}: training dates lack one or more classes: {encoder.classes_}")

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
            objective="multi:softprob", num_class=len(class_names),
            eval_metric="mlogloss", random_state=RANDOM_STATE,
            n_jobs=4, tree_method="hist", verbosity=0,
        )),
    ])
    search = GridSearchCV(pipe, PARAM_GRID, scoring="f1_macro", cv=folds,
                          n_jobs=1, refit=True, verbose=0)
    search.fit(task_train[FEATURES], y_train)
    model = search.best_estimator_
    pred_codes = model.predict(task_test[FEATURES]).astype(int)
    proba = model.predict_proba(task_test[FEATURES])
    pred = encoder.inverse_transform(pred_codes)
    truth = task_test[target_col].astype(str).to_numpy()

    majority_class = task_train[target_col].astype(str).value_counts().idxmax()
    majority_pred = np.repeat(majority_class, len(task_test))
    persistence = task_test[persistence_col].fillna(majority_class).astype(str).to_numpy()

    models[task_name], encoders[task_name] = model, encoder
    predictions[task_name], probabilities[task_name] = (pred, proba)
    reports[task_name] = classification_report(
        truth, pred, labels=encoder.classes_, target_names=encoder.classes_,
        output_dict=True, zero_division=0,
    )
    cm = confusion_matrix(truth, pred, labels=encoder.classes_)
    confusion_tables[task_name] = pd.DataFrame(
        cm, index=[f"Actual: {c}" for c in encoder.classes_],
        columns=[f"Predicted: {c}" for c in encoder.classes_],
    )

    row = {
        "Task": task_name, "Test rows": len(truth),
        "Model accuracy": accuracy_score(truth, pred),
        "Model macro F1": f1_score(truth, pred, average="macro", zero_division=0),
        "Majority accuracy": accuracy_score(truth, majority_pred),
        "Majority macro F1": f1_score(truth, majority_pred, average="macro", zero_division=0),
        "Persistence accuracy": accuracy_score(truth, persistence),
        "Persistence macro F1": f1_score(truth, persistence, average="macro", zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(truth, pred),
        "Macro precision": precision_score(truth, pred, average="macro", zero_division=0),
        "Macro recall": recall_score(truth, pred, average="macro", zero_division=0),
        "Log loss": log_loss(y_test, proba, labels=np.arange(len(encoder.classes_))),
        "Multiclass Brier score": np.mean(np.sum(
            (proba - np.eye(len(encoder.classes_))[y_test]) ** 2, axis=1
        )),
        "Best parameters": str(search.best_params_),
    }
    if task_name == "2-class":
        positive_index = int(np.flatnonzero(encoder.classes_ == "Polluted")[0])
        positive_truth = (truth == "Polluted").astype(int)
        positive_score = proba[:, positive_index]
        row["Polluted ROC-AUC"] = roc_auc_score(positive_truth, positive_score)
        row["Polluted PR-AUC"] = average_precision_score(positive_truth, positive_score)
        row["Binary Brier score"] = brier_score_loss(positive_truth, positive_score)
    summary_rows.append(row)
    print(f"{task_name}: best CV macro F1={search.best_score_:.4f}; params={search.best_params_}")

summary = pd.DataFrame(summary_rows).set_index("Task")
display(summary.round(4))


## 6. Per-class precision, recall, and F1

These reports show whether the model handles rare AQI categories as well as common ones. Macro F1 weights each class equally; support shows how many held-out examples each score is based on.


In [ ]:
for task_name, report in reports.items():
    print(f"\n{task_name} per-class report")
    display(pd.DataFrame(report).T.round(4))


## 7. Confusion matrices

Rows are actual classes and columns are predicted classes. Off-diagonal counts reveal the categories the model confuses.


In [ ]:
for task_name, confusion in confusion_tables.items():
    print(task_name)
    display(confusion)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(
        confusion_matrix=confusion.to_numpy(),
        display_labels=encoders[task_name].classes_,
    ).plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
    ax.set_title(f"{task_name} next-day AQI confusion matrix")
    plt.tight_layout()
    plt.show()


## 8. Binary clean-vs-polluted discrimination and calibration

ROC-AUC and PR-AUC measure how well the binary model ranks polluted days across decision thresholds. PR-AUC is especially informative when the positive class is less common. The reliability plot compares predicted polluted probabilities with observed polluted frequency; Brier score and log loss are probability-quality measures (lower is better).


In [ ]:
binary_task = "2-class"
binary_test = test.dropna(subset=[TASKS[binary_task][0]]).copy()
binary_target = binary_test[TASKS[binary_task][0]].astype(str).to_numpy()
binary_encoder = encoders[binary_task]
binary_truth = (binary_target == "Polluted").astype(int)
binary_proba = probabilities[binary_task]
polluted_index = int(np.flatnonzero(binary_encoder.classes_ == "Polluted")[0])
polluted_score = binary_proba[:, polluted_index]

print(f"ROC-AUC: {roc_auc_score(binary_truth, polluted_score):.4f}")
print(f"PR-AUC:  {average_precision_score(binary_truth, polluted_score):.4f}")
print(f"Brier score: {brier_score_loss(binary_truth, polluted_score):.4f}")
print(f"Log loss: {log_loss(binary_truth, binary_proba, labels=np.arange(len(binary_encoder.classes_))):.4f}")

from sklearn.calibration import calibration_curve
observed_rate, mean_probability = calibration_curve(
    binary_truth, polluted_score, n_bins=10, strategy="quantile"
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.plot(mean_probability, observed_rate, marker="o", label="XGBoost")
ax.set(xlabel="Mean predicted probability of polluted",
       ylabel="Observed polluted frequency",
       title="Clean-vs-polluted probability calibration", xlim=(0, 1), ylim=(0, 1))
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Read the results carefully

- **Accuracy** is the share of all predictions that are correct.
- **Macro F1** gives each class equal weight and helps expose weak performance on rare categories.
- **Majority baseline** always predicts the most common training class; the model should beat it.
- **Persistence baseline** predicts that the current AQI class will continue to tomorrow; it checks whether the model adds value beyond simple day-to-day continuity.
- **Balanced accuracy** averages recall across classes.
- **ROC-AUC, PR-AUC, calibration, Brier score, and log loss** assess binary ranking and probability quality; they do not replace thresholded accuracy or class-level scores.

A high accuracy alone does not establish that the model meets a useful performance target. Review the test period, class distribution, macro F1, baselines, and confusion matrix together.
